In [1]:
import pandas as pd
import numpy as np
import os
import gc
from sklearn.model_selection import train_test_split
import csv
from scipy.stats import t

import heapq

from tqdm import tqdm
tqdm.pandas()


In [2]:
SPLITS = [
    {"split": 1, "train":[1,2,3,4], "test":5},
    {"split": 2, "train":[1,2,3,5], "test":4},
    {"split": 3, "train":[1,2,4,5], "test":3},
    {"split": 4, "train":[1,3,4,5], "test":2},
    {"split": 5, "train":[2,3,4,5], "test":1},
]

CLASS_BALANCE = 10

os.makedirs("splits", exist_ok=True)

In [3]:
df_municipio = pd.read_pickle("IpumsNORWAY_distinct_municipio_emb.pkl")
df_municipio = df_municipio[["MUNI_ID", "emb"]]
municipio_map = df_municipio.set_index("MUNI_ID").to_dict("index")
municipios = list(municipio_map.keys())
municipio_index = {m: i for i, m in enumerate(municipios)}

municipio_emb = np.stack([
    municipio_map[m]["emb"]
    for m in municipios
])

In [4]:
df_profissao_emb = pd.read_pickle("IpumsNORWAY_distinct_profissao_emb.pkl")
df_profissao_emb = df_profissao_emb[["OCC_ID", "emb"]]
profissao_emb_map = df_profissao_emb.set_index("OCC_ID").to_dict("index")

occs = list(profissao_emb_map.keys())
occ_index = {o: i for i, o in enumerate(occs)}

occ_emb = np.stack([
    profissao_emb_map[o]["emb"]
    for o in occs
])

In [5]:
df_person = pd.read_csv("IpumsNORWAY_distinct_person.csv")
df_person = df_person[["person_id","AGE","MUNICNO","BPLNO","OCC_ID"]]
df_person = df_person.replace({np.nan: None})

person_map = {}

for _, r in df_person.iterrows():

    munic_idx = municipio_index.get(r.MUNICNO)
    bpl_idx = municipio_index.get(r.BPLNO)
    occ_idx = occ_index.get(r.OCC_ID)

    person_map[r.person_id] = (
        r.person_id,
        r.AGE,
        munic_idx,
        bpl_idx,
        occ_idx
    )

In [6]:
def calcular_diff_idade(df):
    df = df.merge(
        df_person[["person_id", "AGE"]].rename(columns={"person_id": "person_id_chefe", "AGE": "age_chefe"}),
        on="person_id_chefe",
        how="left"
    )
    
    df = df.merge(
        df_person[["person_id", "AGE"]].rename(columns={"person_id": "person_id_parente", "AGE": "age_parente"}),
        on="person_id_parente",
        how="left"
    )
    
    df["age_diff"] = df["age_chefe"] - df["age_parente"]
    return df

In [7]:
def estimar_t_student_parameters (df):

    related_types = df.RELATED.unique()
    t_student_parameters = {}
    for related_type in related_types:
    
        
        # Dados de treino: diferença de idade por tipo de relação de parentesco
        dados = df[df.RELATED == related_type].age_diff.dropna().to_list()
        
        # Estimação parametros da t-Student
        graus_liberdade, localizacao, escala = t.fit(dados)
        
        t_student_parameters[related_type] = {
            "graus_liberdade": graus_liberdade,
            "localizacao": localizacao,
            "escala": escala
        }
    return t_student_parameters

In [8]:
# função para score de endereço
def addr_score_fast(m1, m2, m3, m4):

    best = -1e9

    if m1 is not None and m3 is not None:
        best = max(best, municipio_emb[m1] @ municipio_emb[m3])

    if m1 is not None and m4 is not None:
        best = max(best, municipio_emb[m1] @ municipio_emb[m4])

    if m2 is not None and m3 is not None:
        best = max(best, municipio_emb[m2] @ municipio_emb[m3])

    if m2 is not None and m4 is not None:
        best = max(best, municipio_emb[m2] @ municipio_emb[m4])

    return best

In [9]:
# gerar par
def gerar_par(ID, RELATED, chefe, parente_id, cand, related, t_dist, t_height):

    chefe_id, chefe_age, chefe_munic, chefe_bpl, chefe_occ = chefe
    cand_id, cand_age, cand_munic, cand_bpl, cand_occ = cand

    label = int(cand_id == parente_id)

    # AGE
    if chefe_age is not None and cand_age is not None:

        missing_age = 0

        diff = chefe_age - cand_age

        dist = t_dist[related]

        altura = dist.pdf(diff)

        age_score = altura / t_height[related]

    else:

        missing_age = 1
        age_score = None

    # ADDR

    if (chefe_munic is not None or chefe_bpl is not None) and \
       (cand_munic is not None or cand_bpl is not None):

        missing_addr = 0

        addr_score = addr_score_fast(
            chefe_munic,
            chefe_bpl,
            cand_munic,
            cand_bpl
        )

        addr_exato = int(
            chefe_munic == cand_munic or
            chefe_munic == cand_bpl or
            chefe_bpl == cand_munic or
            chefe_bpl == cand_bpl
        )

    else:

        missing_addr = 1
        addr_score = None
        addr_exato = None

    # OCC

    if chefe_occ is not None and cand_occ is not None:

        missing_occ = 0

        occ_score = occ_emb[chefe_occ] @ occ_emb[cand_occ]

    else:

        missing_occ = 1
        occ_score = None

    return (
        ID,
        RELATED,
        chefe_id,
        cand_id,
        label,
        missing_age,
        age_score,
        missing_addr,
        addr_score,
        addr_exato,
        missing_occ,
        occ_score
    )

In [10]:
# processar linha
def processar_linha(row, t_dist, t_height, class_balance):
    chefe = person_map[row.person_id_chefe]
    chefe_id, chefe_age, chefe_munic, chefe_bpl, chefe_occ = chefe
    parente_id = row.person_id_parente

    prioridade = []

    candidates = row.lista_homonimos
    
    if class_balance != None:
        for id_homonimo in row.lista_homonimos:
            cand_id, cand_age, cand_munic, cand_bpl, cand_occ = person_map[id_homonimo]
            score = 1
            if cand_id == parente_id:
                score = 9
            else :
                both_age = chefe_age is not None and cand_age is not None
        
                both_addr = (
                    ((chefe_munic is not None) or (chefe_bpl is not None))
                    and
                    ((cand_munic is not None) or (cand_bpl is not None))
                )
        
                both_occ = (
                    chefe_occ not in (None, "__")
                    and
                    cand_occ not in (None, "__")
                )
        
                if both_age and both_addr and both_occ:
                    score = 4
                elif (both_age and both_addr) or (both_age and both_occ):
                    score = 3
                elif both_occ and both_addr:
                    score = 2
                elif both_age or both_addr or both_occ:
                    score = 1
    
            prioridade.append((score, cand_id))
    
        candidates = heapq.nlargest(class_balance + 1, prioridade)
        candidates = [cand_id for _, cand_id in candidates]
    
    pares = []
    for cand_id in candidates:
            cand = person_map[cand_id]
    
            pares.append(
                gerar_par(
                    row.ID,
                    row.RELATED,
                    chefe,
                    parente_id,
                    cand,
                    row.RELATED,
                    t_dist, t_height
                )
            )
    
    return pares

In [11]:
def processar_df(df, file_name, class_balance):
    rows = df.itertuples(index=False)

    OUTPUT_FILE = f"{file_name}.csv"
    field_names = ["ID", "RELATED", "person_id_chefe", "person_id_candidato", "label", "missing_age", "age_score", "missing_addr", "addr_score", "addr_exato", "missing_occ", "occ_score"]
    
    rows = df.itertuples(index=False)
    with open(OUTPUT_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(field_names)
    
        def process_row(row):
            pares = processar_linha(row, t_dist, t_height, class_balance)
            if pares and len(pares) >0:    
                writer.writerows(pares)
    
            
        for row in tqdm(rows, total=len(df)):
            process_row(row)

In [12]:
df_pares = pd.read_pickle("IpumsNORWAY_pares.pkl")

In [13]:
for split in SPLITS:

    split_dir = f"splits/{split['split']}"
    os.makedirs(split_dir, exist_ok=True)
    
    
    df_train_val = df_pares[df_pares["FOLD"].isin(split["train"])]
    df_train, df_val = train_test_split(
        df_train_val, 
        test_size=0.10,
        random_state=42,
        shuffle=True
    )
    df_test = df_pares[df_pares["FOLD"] == split["test"]]
    
    df_train = calcular_diff_idade(df_train)
    df_val = calcular_diff_idade(df_val)
    df_test = calcular_diff_idade(df_test)
    
    t_student_parameters = estimar_t_student_parameters(df_train)
    # pré-compute distribuição t
    t_dist = {}
    t_height = {}
    for r, p in t_student_parameters.items():
    
        dist = t(
            df=p["graus_liberdade"],
            loc=p["localizacao"],
            scale=p["escala"]
        )
    
        t_dist[r] = dist
        t_height[r] = dist.pdf(p["localizacao"])
    
    
    # Gerar arquivos de pares
    processar_df(df_train, f"{split_dir}/train_pares", class_balance=CLASS_BALANCE)
    processar_df(df_val, f"{split_dir}/val_pares", class_balance=CLASS_BALANCE)
    processar_df(df_test, f"{split_dir}/test_pares", class_balance=None)


100%|██████████| 156777/156777 [1:26:01<00:00, 30.37it/s]  
